<a href="https://githubtocolab.com/giswqs/gee-tutorials/blob/master/AssetManagement/extract_values.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Uncomment the following line to install [geemap](https://geemap.org) if needed.

In [ ]:
# !pip install geemap

In [ ]:
# import geemap
# geemap.update_package()

# Interactive extraction of pixel values and interactive region reduction

## Interactive extraction of pixel values

### Import libraries

In [1]:
import os
import ee
import geemap
import geopandas as gpd

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
def mask_landsat_sr_and_calculate_indices(image):
    # Define cloud and cloud shadow bitmasks
    cloud_shadow_bitmask = 1 << 3  # Bit 3 is for cloud shadow
    clouds_bitmask = 1 << 4        # Bit 4 is for clouds

    # Select the QA_PIXEL band
    qa = image.select('QA_PIXEL')
    
    # Create the mask
    mask = qa.bitwiseAnd(cloud_shadow_bitmask).eq(0).And(
        qa.bitwiseAnd(clouds_bitmask).eq(0)
    )
    
    # Apply the mask
    image = image.updateMask(mask)
    
    # Calculate NDVI = (NIR - Red) / (NIR + Red)
    ndvi = image.normalizedDifference(['B4', 'B3']).rename('NDVI')
    
    # Calculate NDMI = (NIR - SWIR1) / (NIR + SWIR1)
    ndmi = image.normalizedDifference(['B4', 'B5']).rename('NDMI')
    
    # Calculate SAVI = ((NIR - Red) / (NIR + Red + L)) * (1 + L)
    # L is a soil brightness correction factor, often set to 0.5
    nir = image.select('B4')
    red = image.select('B3')
    savi = nir.subtract(red).divide(nir.add(red).add(0.5)).multiply(1.5).rename('SAVI')
    
    # Add NDVI, NDMI, and SAVI as new bands to the image
    return image.addBands([ndvi, ndmi, savi]).select(["B1", "B2", "B3", "B4", "B5", "B7", "NDVI", "NDMI", "SAVI"])

In [ ]:
ee.Initialize()
shapefile_path = "intersected_pixel_points.shp"  # Update with the path to your shapefile
shapefile = gpd.read_file(shapefile_path)

# Convert the shapefile bounds to a GeoJSON geometry
aoi = geemap.geopandas_to_ee(shapefile)
start_date = "2012-12-01"
end_date = "2013-03-31"

landsat = ee.ImageCollection("LANDSAT/LE07/C01/T1_SR") \
    .filterDate(start_date, end_date) \
    .filterBounds(aoi) \
    .map(mask_landsat_sr_and_calculate_indices)

# Median composite with NDVI, NDMI, SAVI
median_image = landsat.median()

In [ ]:
points = median_image.sample(
    region=aoi.geometry(),  # Region of interest
    scale=30,  # Spatial resolution of Landsat 7
    projection='EPSG:4326',  # Use latitude/longitude projection
    geometries=True  # Include lat/lon as geometry
)

In [ ]:
Map = geemap.Map()

image = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318').select(
    ['B5', 'B4', 'B3']
)

vis_params = {'min': 0, 'max': 0.5, 'gamma': [0.95, 1.1, 1]}

Map.centerObject(image, 8)
Map.addLayer(image, vis_params, 'Landsat')
Map

region = ee.Geometry.BBox(-122.5955, 37.5339, -122.0982, 37.8252)
fc = ee.FeatureCollection(region)
style = {'color': 'ffff00ff', 'fillColor': '00000000'}
Map.addLayer(fc.style(**style), {}, 'ROI')
Map

In [ ]:
geemap.ee_export_image(image, filename="landsat.tif", scale=30, region=region)

In [ ]:
shapefile_path = "/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion.shp"  # Update with your shapefile path
gdf = gpd.read_file(shapefile_path)


# Convert GeoDataFrame to an Earth Engine object
aoi = geemap.geopandas_to_ee(gdf)

# Define the date range and get the Landsat 7 ImageCollection
start_date = "2012-12-01"
end_date = "2012-12-16"
landsat = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2") \
    .filterDate(start_date, end_date) \
    .filterBounds(aoi)

# Apply cloud masking and calculate indices
def mask_landsat_sr(image):
    cloud_shadow_bitmask = 1 << 3  # Bit 3 is for cloud shadow
    clouds_bitmask = 1 << 4        # Bit 4 is for clouds
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(cloud_shadow_bitmask).eq(0).And(
        qa.bitwiseAnd(clouds_bitmask).eq(0)
    )
    #return image.updateMask(mask).select(["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"])
    #return image.updateMask(mask).select(["SR_B3"])
    return image.select(["SR_B3"])

landsat = landsat.map(mask_landsat_sr)

# Create a median composite image
#median_image = landsat.median()
median_image = landsat.max()
# Visualization parameters
#vis_params = {'min': 0, 'max': 3000, 'bands': ['SR_B4', 'SR_B3', 'SR_B2']}
vis_params = {'min': 0, 'max': 3000, 'bands': ['SR_B3']}
# Initialize the map
Map = geemap.Map()

# Center the map on the AOI
Map.centerObject(aoi, 8)

# Add the median image to the map
Map.addLayer(median_image, vis_params, 'Median Composite')

# Add the AOI to the map with a yellow outline
style = {'color': 'ffff00ff', 'fillColor': '00000000'}
Map.addLayer(aoi.style(**style), {}, 'ROI')

# Display the map
Map

In [ ]:
aoi_grid = aoi.geometry().coveringGrid(ee.Projection('EPSG:4326'), scale=1000)  # 1000-meter grid cells
aoi_grid.size().getInfo()
# Process each grid cell
# for i in range(aoi_grid.size().getInfo()):
#     ##print(i)
#     subregion = aoi_grid.get(i)
#     points = median_image.sample(
#         region=subregion,
#         scale=30,
#         projection='EPSG:4326',
#         geometries=True
#     )
#     task = ee.batch.Export.table.toDrive(
#         collection=points,
#         description=f"LandsatMedianImage_Part{i}",
#         folder="GEE",
#         fileNamePrefix=f"landsat_pixel_values_part{i}",
#         fileFormat="CSV"
#     )
#     task.start()

In [ ]:
# Export the clipped image
clipped_image = median_image.clip(aoi)

task = ee.batch.Export.image.toDrive(
    image=clipped_image,
    description="LandsatMedianImage",  # Task description
    folder="GEE",  # Folder name in Google Drive
    fileNamePrefix="median_image",  # File name prefix
    scale=30,  # Spatial resolution in meters
    region=aoi.geometry().getInfo(),  # AOI as GeoJSON geometry
    maxPixels=1e13  # Adjust for large images
)

# Start the export task
task.start()

In [ ]:
# Step 4: Calculate the median composite
median_image = landsat.median()

# Step 5: Extract median values at each point
result = geemap.extract_values_to_points(aoi, median_image, scale=30)

# Convert result to a GeoDataFrame
median_gdf = geemap.ee_to_geopandas(result)

# Save results as a new shapefile
output_path = "median_values.shp"
median_gdf.to_file(output_path)
print(f"Median values saved to {output_path}")

In [ ]:
ee.Initialize()

# Define the coordinates of the point (lon, lat)
point = ee.Geometry.Point([113.633324, 22.597788])  # Note: (longitude, latitude)

# Filter the Landsat 8 Image Collection to the area of interest
landsat_collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA') \
    .filterBounds(point) \
    .filterDate('2019-01-01', '2019-05-31') \
    .select(['B4', 'B5'])  # Example: Select Red (B4) and NIR (B5) bands

# Extract the pixel value at the given point for each image in the collection
def extract_pixel_value(image):
    value = image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30  # Scale of Landsat 8
    )
    return image.set(value)  # Attach the pixel value as properties of the image

# Map the function over the image collection
time_series = landsat_collection.map(extract_pixel_value)

# Extract the pixel value and date from each image in the collection
values = time_series.reduceColumns(
    ee.Reducer.toList(2),  # 2 properties: date and band value
    ['system:time_start', 'B5']  # Change 'B4' to the band you want to track
).get('list')

# Get the result as a Python list
results = ee.List(values).getInfo()

# Print the time series
for entry in results:
    date = ee.Date(entry[0]).format('YYYY-MM-dd').getInfo()
    pixel_value = entry[1]
    print(f"Date: {date}, Pixel Value: {pixel_value}")

In [ ]:
landsat7 = ee.Image('LE7_TOA_5YEAR/1999_2003') \
    .select([0, 1, 2, 3, 4, 6])
landsat_vis = {
    'bands': ['B4', 'B3', 'B2'], 
    'gamma': 1.7
}
Map.addLayer(landsat7, landsat_vis, "LE7_TOA_5YEAR/1999_2003")

Map.set_plot_options(add_marker_cluster=True)

### Activate the plotting tool

Tick the `Plotting` checkbox and click the mouse on the map to start displaying charts.

### Export pixel values to shapefile/csv

In [ ]:
out_dir = os.path.expanduser('~/Downloads')
# out_csv = os.path.join(out_dir, 'points.csv')
out_shp = os.path.join(out_dir, 'points.shp')

In [ ]:
Map.extract_values_to_points(out_shp)

## Interactive Region Reduction

### Import libraries

In [ ]:
import geemap.colormaps as cm

### Create an interactive map

In [ ]:
m = geemap.Map()

### Add add to the map

In [ ]:
collection = ee.ImageCollection('MODIS/006/MOD13A2') \
    .filterDate('2015-01-01', '2019-12-31') \
    .select('NDVI')

# Convert the image collection to an image.
image = collection.toBands()

palette = cm.palettes.ndvi

ndvi_vis = {
  'min': 0.0,
  'max': 9000.0,
  'palette': palette
}

m.addLayer(image, {}, 'MODIS NDVI Time-series')
m.addLayer(image.select(0), ndvi_vis, 'MODIS NDVI VIS')

m

### Set reducer

In [ ]:
m.set_plot_options(add_marker_cluster=True, marker=None)
m.roi_reducer = ee.Reducer.mean()

### Export data

In [ ]:
out_dir = os.path.expanduser('~/Downloads')
# out_csv = os.path.join(out_dir, 'points.csv')
out_shp = os.path.join(out_dir, 'ndvi.shp')
m.extract_values_to_points(out_shp)

## Creating training samples

In [ ]:
import geemap

In [ ]:
geemap.show_youtube("https://youtu.be/VWh5PxXPZw0")

In [ ]:
Map = geemap.Map()
Map

In [ ]:
Map = geemap.Map()
Map